# 03 — Feature Sanity (Phase 2)

Visual sanity checks for the leakage-safe feature pipeline:

1. Plot each Phase 2 feature (RSI, MACD) over time against price.
2. Correlation matrix between all technical features — flag near-duplicates.
3. Spot-check 2–3 rows of computed features against raw prices by hand.

This notebook contains **no logic** — all computation lives in
`src/features/`. It only reads the built parquet and visualises.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})

In [ ]:
INTERVAL = "1d"
FEATURES_PATH = Path(f"data/processed/features_{INTERVAL}.parquet")
RAW_PATH = Path(f"data/raw/btc_usdt_{INTERVAL}.parquet")
MANIFEST_PATH = Path(f"data/processed/features_{INTERVAL}.manifest.json")

features = pd.read_parquet(FEATURES_PATH)
raw = pd.read_parquet(RAW_PATH)
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

print(f"Features: {features.shape[0]} rows × {features.shape[1]} columns")
print(f"Raw candles: {raw.shape[0]} rows")
print(f"Feature families: {list(manifest['features'].keys())}")
print(f"NaN rows: {manifest['nan_report']['rows_with_any_nan']}")

## 1. RSI over time vs price

In [ ]:
# Find the RSI column (name depends on rsi_period from config).
rsi_cols = [c for c in features.columns if c.startswith("rsi_")]
assert rsi_cols, "No RSI column found"
rsi_col = rsi_cols[0]

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(14, 8))

ax1.plot(features["open_time"], raw["close"], linewidth=0.7, label="close")
ax1.set_ylabel("BTC/USDT close")
ax1.legend()
ax1.set_title(f"Close price and {rsi_col} — {INTERVAL}")

ax2.plot(features["open_time"], features[rsi_col], linewidth=0.7,
         color="tab:orange", label=rsi_col)
ax2.axhline(70, color="red", linestyle="--", alpha=0.5, label="overbought (70)")
ax2.axhline(30, color="green", linestyle="--", alpha=0.5, label="oversold (30)")
ax2.set_ylabel("RSI")
ax2.set_ylim(0, 100)
ax2.legend()

plt.tight_layout()
plt.show()

## 2. MACD over time vs price

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(14, 8))

ax1.plot(features["open_time"], raw["close"], linewidth=0.7, label="close")
ax1.set_ylabel("BTC/USDT close")
ax1.legend()
ax1.set_title(f"Close price and MACD — {INTERVAL}")

ax2.plot(features["open_time"], features["macd_line"],
         linewidth=0.7, label="MACD line")
ax2.plot(features["open_time"], features["macd_signal"],
         linewidth=0.7, label="signal")
colors = ["tab:green" if v >= 0 else "tab:red"
          for v in features["macd_hist"]]
ax2.bar(features["open_time"], features["macd_hist"],
        color=colors, alpha=0.4, width=1, label="histogram")
ax2.axhline(0, color="black", linewidth=0.5)
ax2.set_ylabel("MACD")
ax2.legend()

plt.tight_layout()
plt.show()

## 3. Correlation matrix

In [ ]:
# Technical features only (exclude open_time and CDL* integer columns).
tech_cols = [c for c in features.columns
             if c != "open_time" and not c.startswith("cdl_")]

corr = features[tech_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(tech_cols)))
ax.set_yticks(range(len(tech_cols)))
ax.set_xticklabels(tech_cols, rotation=90, fontsize=7)
ax.set_yticklabels(tech_cols, fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Feature correlation matrix (technical only)")
plt.tight_layout()
plt.show()

# Flag near-duplicates: |r| > 0.95.
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
high = corr.where(mask).stack()
near_dup = high[high.abs() > 0.95].sort_values(ascending=False)
if len(near_dup):
    print("\n\u26a0 Near-duplicate features (|r| > 0.95):")
    print(near_dup.to_string())
else:
    print("\n\u2713 No near-duplicate feature pairs (|r| \u2264 0.95).")

## 4. Spot-check: manually verify feature rows against raw data

In [ ]:
# Pick rows 50, 100, 200 for spot-checking.
SPOT_ROWS = [50, 100, 200]

for t in SPOT_ROWS:
    if t >= len(raw):
        print(f"  Row {t}: out of range (only {len(raw)} rows)")
        continue
    print(f"\n=== Row {t} ===")
    print(f"  open_time:  {raw['open_time'].iloc[t]}")
    print(f"  O/H/L/C:   {raw['open'].iloc[t]:.2f} / "
          f"{raw['high'].iloc[t]:.2f} / "
          f"{raw['low'].iloc[t]:.2f} / "
          f"{raw['close'].iloc[t]:.2f}")
    print(f"  volume:     {raw['volume'].iloc[t]:.2f}")

    # Hand-check log_ret_1: log(close[t] / close[t-1]).
    if t >= 1:
        expected_lr1 = np.log(
            raw['close'].iloc[t] / raw['close'].iloc[t - 1]
        )
        actual_lr1 = features['log_ret_1'].iloc[t]
        match = '\u2713' if np.isclose(expected_lr1, actual_lr1) else '\u2717'
        print(f"  log_ret_1:  computed={actual_lr1:.8f}, "
              f"expected={expected_lr1:.8f}  {match}")

    # Hand-check RSI value (just display for manual inspection).
    print(f"  {rsi_col}:      {features[rsi_col].iloc[t]:.4f}")
    print(f"  macd_line:  {features['macd_line'].iloc[t]:.4f}")
    print(f"  macd_hist:  {features['macd_hist'].iloc[t]:.4f}")